# 10 — Grad-CAM interpretation

Visualise which (Doppler, time) regions drove the best model's PD vs control
decisions. Aggregating Grad-CAM maps across true positives surfaces the
population-level features the network learned to attend to.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

import torch

from src.models import resnet18_finetune
from src.dataset import WindowDataset
from src.interpret import GradCAM, upsample_heatmap_to


## 1. Load a trained model checkpoint

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = ROOT / "outputs" / "models" / "resnet_best.pt"

model = resnet18_finetune(in_channels=2, pretrained=False)
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"loaded {ckpt_path}")
else:
    print(f"!! no checkpoint at {ckpt_path} — re-run Notebook 09 with checkpoint saving enabled")
model = model.eval().to(device)


## 2. Grab a few PD windows for inspection

In [ ]:
manifest = pd.read_csv(ROOT / "outputs" / "preprocessed" / "manifest.csv")
cache_root = ROOT / "outputs" / "preprocessed"
pd_rows = manifest[manifest["label"] == 1].sample(6, random_state=0)
pd_rows


## 3. Compute Grad-CAM per window

In [ ]:
cam = GradCAM(model, model.layer4[-1].conv2)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, (_, row) in enumerate(pd_rows.iterrows()):
    arr = np.load(cache_root / row["npy_path"]).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).to(device)
    heatmap = cam(x, class_idx=1)
    heatmap = upsample_heatmap_to(heatmap, arr.shape[1:])

    axes[0, i].imshow(arr[0], aspect="auto", origin="lower", cmap="magma")
    axes[0, i].set_title(f"{row['subject_id']}", fontsize=8)
    axes[1, i].imshow(arr[0], aspect="auto", origin="lower", cmap="gray", alpha=0.7)
    axes[1, i].imshow(heatmap, aspect="auto", origin="lower", cmap="jet", alpha=0.5)

axes[0, 0].set_ylabel("input foot")
axes[1, 0].set_ylabel("Grad-CAM overlay")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("PD windows — Grad-CAM (target class = PD)")
plt.tight_layout(); plt.show()
cam.close()


### Reading the maps

- High activation in the **foot Doppler band (≥ 200 Hz)** = the model is using foot-strike content to decide PD.
- High activation in the **torso band (0–200 Hz)** = trunk sway / cadence.
- If maps look noisy / uniform, the model may not have learned a consistent pattern, or the layer choice is too deep — try `model.layer3[-1].conv2`.

Aggregate the heatmaps across many true-positive windows to denoise individual-window variability:


In [ ]:
cam = GradCAM(model, model.layer4[-1].conv2)
acc = None; n = 0
sample = manifest[manifest["label"] == 1].sample(50, random_state=0)
for _, row in sample.iterrows():
    arr = np.load(cache_root / row["npy_path"]).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).to(device)
    h = cam(x, class_idx=1)
    h = upsample_heatmap_to(h, arr.shape[1:])
    acc = h if acc is None else acc + h
    n += 1
mean_heatmap = acc / max(n, 1)
cam.close()

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(mean_heatmap, aspect="auto", origin="lower", cmap="jet")
fig.colorbar(im, ax=ax, fraction=0.04)
ax.set_title(f"Mean Grad-CAM over {n} PD windows")
ax.set_xlabel("time bin"); ax.set_ylabel("Doppler bin")
plt.tight_layout(); plt.show()
